In [4]:
import heapq
from copy import deepcopy
import time

class NPuzzle:
    def __init__(self, n, trang_thai_dau, trang_thai_dich=None):
        self.n = n
        self.trang_thai_dau = trang_thai_dau
        if trang_thai_dich is None:
            # Tạo trạng thái đích: số từ 1 đến n*n-1, sau đó là 0
            self.trang_thai_dich = [[i*n + j + 1 if i*n + j + 1 < n*n else 0
                                     for j in range(n)] for i in range(n)]
        else:
            self.trang_thai_dich = trang_thai_dich

    def tim_vi_tri_trong(self, state):
        """Tìm vị trí ô trống (0)"""
        for i in range(self.n):
            for j in range(self.n):
                if state[i][j] == 0:
                    return i, j
        return None

    def tinh_heuristic_manhattan(self, state):
        """Heuristic khoảng cách Manhattan"""
        khoang_cach = 0
        for i in range(self.n):
            for j in range(self.n):
                gia_tri = state[i][j]
                if gia_tri != 0:
                    dich_i = (gia_tri - 1) // self.n
                    dich_j = (gia_tri - 1) % self.n
                    khoang_cach += abs(i - dich_i) + abs(j - dich_j)
        return khoang_cach

    def tinh_heuristic_sai_vi_tri(self, state):
        """Heuristic số ô sai vị trí"""
        sai_vi_tri = 0
        for i in range(self.n):
            for j in range(self.n):
                if state[i][j] != 0 and state[i][j] != self.trang_thai_dich[i][j]:
                    sai_vi_tri += 1
        return sai_vi_tri

    def tinh_heuristic_xung_dot_tuyen(self, state):
        """Heuristic xung đột tuyến tính (cải tiến của Manhattan)"""
        khoang_cach = self.tinh_heuristic_manhattan(state)

        # Kiểm tra xung đột trên hàng
        for i in range(self.n):
            hang = [state[i][j] for j in range(self.n) if state[i][j] != 0]
            for j in range(len(hang)):
                for k in range(j + 1, len(hang)):
                    if (hang[j] - 1) // self.n == i and (hang[k] - 1) // self.n == i:
                        if hang[j] > hang[k]:
                            khoang_cach += 2

        # Kiểm tra xung đột trên cột
        for j in range(self.n):
            cot = [state[i][j] for i in range(self.n) if state[i][j] != 0]
            for i in range(len(cot)):
                for k in range(i + 1, len(cot)):
                    if (cot[i] - 1) % self.n == j and (cot[k] - 1) % self.n == j:
                        if cot[i] > cot[k]:
                            khoang_cach += 2

        return khoang_cach

    def lay_trang_thai_ke(self, state):
        """Lấy các trạng thái lân cận"""
        trang_thai_ke = []
        trong_i, trong_j = self.tim_vi_tri_trong(state)

        cac_huong = [(-1, 0), (1, 0), (0, -1), (0, 1)]

        for di, dj in cac_huong:
            moi_i, moi_j = trong_i + di, trong_j + dj
            if 0 <= moi_i < self.n and 0 <= moi_j < self.n:
                trang_thai_moi = deepcopy(state)
                trang_thai_moi[trong_i][trong_j], trang_thai_moi[moi_i][moi_j] = \
                    trang_thai_moi[moi_i][moi_j], trang_thai_moi[trong_i][trong_j]
                trang_thai_ke.append(trang_thai_moi)

        return trang_thai_ke

    def doi_sang_tuple(self, state):
        """Chuyển state thành tuple để hash"""
        return tuple(tuple(row) for row in state)

    def giai_akt(self, heuristic='xung_dot_tuyen', max_trang_thai=1000000):
        """Giải N-Puzzle bằng thuật toán A*"""
        thoi_gian_bat_dau = time.time()

        # Chọn heuristic
        if heuristic == 'manhattan':
            ham_h = self.tinh_heuristic_manhattan
        elif heuristic == 'sai_vi_tri':
            ham_h = self.tinh_heuristic_sai_vi_tri
        else:
            ham_h = self.tinh_heuristic_xung_dot_tuyen

        # Nút bắt đầu
        nut_bat_dau = {
            'state': self.trang_thai_dau,
            'g': 0,
            'h': ham_h(self.trang_thai_dau),
            'parent': None
        }
        nut_bat_dau['f'] = nut_bat_dau['g'] + nut_bat_dau['h']

        open_set = [(nut_bat_dau['f'], id(nut_bat_dau), nut_bat_dau)]
        closed_set = set()
        g_values = {self.doi_sang_tuple(self.trang_thai_dau): 0}

        so_nut_mo_rong = 0

        while open_set and so_nut_mo_rong < max_trang_thai:
            _, _, current = heapq.heappop(open_set)
            current_tuple = self.doi_sang_tuple(current['state'])

            if current_tuple in closed_set:
                continue

            so_nut_mo_rong += 1

            # Kiểm tra đến đích
            if current['state'] == self.trang_thai_dich:
                thoi_gian_chay = time.time() - thoi_gian_bat_dau
                duong_di = self.tao_lai_duong_di(current)
                return {
                    'duong_di': duong_di,
                    'so_buoc': len(duong_di) - 1,
                    'so_nut_mo_rong': so_nut_mo_rong,
                    'thoi_gian': thoi_gian_chay,
                    'thanh_cong': True
                }

            closed_set.add(current_tuple)

            for trang_thai_ke in self.lay_trang_thai_ke(current['state']):
                neighbor_tuple = self.doi_sang_tuple(trang_thai_ke)

                if neighbor_tuple in closed_set:
                    continue

                g_moi = current['g'] + 1

                if neighbor_tuple not in g_values or g_moi < g_values[neighbor_tuple]:
                    g_values[neighbor_tuple] = g_moi
                    nut_ke = {
                        'state': trang_thai_ke,
                        'g': g_moi,
                        'h': ham_h(trang_thai_ke),
                        'parent': current
                    }
                    nut_ke['f'] = nut_ke['g'] + nut_ke['h']
                    heapq.heappush(open_set, (nut_ke['f'], id(nut_ke), nut_ke))

        return {
            'thanh_cong': False,
            'so_nut_mo_rong': so_nut_mo_rong,
            'thoi_gian': time.time() - thoi_gian_bat_dau
        }

    def tao_lai_duong_di(self, node):
        """Tạo lại đường đi"""
        duong_di = []
        while node:
            duong_di.append(node['state'])
            node = node['parent']
        return list(reversed(duong_di))

    def in_loi_giai(self, ket_qua):
        """In kết quả"""
        if not ket_qua['thanh_cong']:
            print(f"Không tìm thấy lời giải sau khi mở rộng {ket_qua['so_nut_mo_rong']} nút")
            print(f"Thời gian: {ket_qua['thoi_gian']:.2f} giây")
            return

        print(f"✓ Tìm thấy lời giải trong {ket_qua['so_buoc']} bước!")
        print(f"Số nút đã mở rộng: {ket_qua['so_nut_mo_rong']}")
        print(f"Thời gian: {ket_qua['thoi_gian']:.2f} giây")

        # Chỉ in nếu số bước ít
        if ket_qua['so_buoc'] <= 30:
            for i, state in enumerate(ket_qua['duong_di'][:15]):
                print(f"\nBước {i}:")
                for row in state:
                    print(' '.join(f'{x:3d}' for x in row))
            if ket_qua['so_buoc'] > 15:
                print(f"\n... và {ket_qua['so_buoc'] - 15} bước tiếp theo")

# Chạy thử nghiệm với n=5
if __name__ == "__main__":
    # Bảng 5x5 (n=5)
    trang_thai_dau_5x5 = [
        [1, 2, 3, 4, 5],
        [6, 7, 8, 9, 10],
        [11, 12, 13, 14, 15],
        [16, 17, 18, 0, 19],
        [21, 22, 23, 24, 20]
    ]

    print("Giải bài toán 5x5 Puzzle (n=5)...")
    print("Trạng thái ban đầu:")
    for row in trang_thai_dau_5x5:
        print(' '.join(f'{x:3d}' for x in row))

    print("\n" + "="*50)

    puzzle = NPuzzle(5, trang_thai_dau_5x5)
    ket_qua = puzzle.giai_akt(heuristic='xung_dot_tuyen', max_trang_thai=500000)
    puzzle.in_loi_giai(ket_qua)

Giải bài toán 5x5 Puzzle (n=5)...
Trạng thái ban đầu:
  1   2   3   4   5
  6   7   8   9  10
 11  12  13  14  15
 16  17  18   0  19
 21  22  23  24  20

✓ Tìm thấy lời giải trong 2 bước!
Số nút đã mở rộng: 3
Thời gian: 0.00 giây

Bước 0:
  1   2   3   4   5
  6   7   8   9  10
 11  12  13  14  15
 16  17  18   0  19
 21  22  23  24  20

Bước 1:
  1   2   3   4   5
  6   7   8   9  10
 11  12  13  14  15
 16  17  18  19   0
 21  22  23  24  20

Bước 2:
  1   2   3   4   5
  6   7   8   9  10
 11  12  13  14  15
 16  17  18  19  20
 21  22  23  24   0


In [5]:
import heapq
from itertools import permutations
import time
import math

class TSP_AStar:
    def __init__(self, khoang_cach):
        """
        khoang_cach: ma trận khoảng cách giữa các thành phố
        khoang_cach[i][j] = khoảng cách từ thành phố i đến j
        """
        self.khoang_cach = khoang_cach
        self.n = len(khoang_cach)
        self.ten_thanh_pho = [f"TP_{i}" for i in range(self.n)]

    def dat_ten_thanh_pho(self, ten):
        """Đặt tên cho các thành phố"""
        self.ten_thanh_pho = ten

    def heuristic_cay_khung(self, cac_thanh_pho_con_lai, thanh_pho_hien_tai):
        """
        Heuristic dùng cây khung nhỏ nhất (MST) cho các thành phố chưa thăm
        Đây là cận dưới của chi phí còn lại
        """
        if not cac_thanh_pho_con_lai:
            return 0

        # Thêm thành phố hiện tại vào tập cần xét
        cac_dinh = [thanh_pho_hien_tai] + list(cac_thanh_pho_con_lai)

        # Thuật toán Prim tìm MST
        chi_phi_mst = 0
        da_tham = {cac_dinh[0]}
        chua_tham = set(cac_dinh[1:])

        while chua_tham:
            canh_nho_nhat = float('inf')
            thanh_pho_nho_nhat = None

            for u in da_tham:
                for v in chua_tham:
                    if self.khoang_cach[u][v] < canh_nho_nhat:
                        canh_nho_nhat = self.khoang_cach[u][v]
                        thanh_pho_nho_nhat = v

            chi_phi_mst += canh_nho_nhat
            da_tham.add(thanh_pho_nho_nhat)
            chua_tham.remove(thanh_pho_nho_nhat)

        return chi_phi_mst

    def heuristic_canh_nho_nhat(self, cac_thanh_pho_con_lai, thanh_pho_hien_tai):
        """
        Heuristic đơn giản: tổng các cạnh nhỏ nhất từ mỗi thành phố còn lại
        """
        if not cac_thanh_pho_con_lai:
            return 0

        tong = 0
        cac_tp = list(cac_thanh_pho_con_lai)

        for i, tp in enumerate(cac_tp):
            # Tìm khoảng cách nhỏ nhất từ tp đến các tp khác
            min_dist = float('inf')
            for tp_khac in cac_tp[i+1:]:
                min_dist = min(min_dist, self.khoang_cach[tp][tp_khac])
            if cac_tp:
                min_dist = min(min_dist, self.khoang_cach[tp][thanh_pho_hien_tai])
            if min_dist != float('inf'):
                tong += min_dist

        return tong / 2  # Chia 2 vì mỗi cạnh được đếm 2 lần

    def giai_tsp(self, thanh_pho_bat_dau=0, heuristic='mst'):
        """
        Giải bài toán TSP bằng thuật toán A*
        Trạng thái: (thanh_pho_hien_tai, tap_da_tham)
        """
        thoi_gian_bat_dau = time.time()

        # Chọn heuristic
        if heuristic == 'mst':
            ham_h = self.heuristic_cay_khung
        else:
            ham_h = self.heuristic_canh_nho_nhat

        # Trạng thái ban đầu
        tap_da_tham = frozenset([thanh_pho_bat_dau])
        nut_bat_dau = {
            'hien_tai': thanh_pho_bat_dau,
            'da_tham': tap_da_tham,
            'duong_di': [thanh_pho_bat_dau],
            'chi_phi': 0,
            'h': ham_h(tap_da_tham, thanh_pho_bat_dau)
        }
        nut_bat_dau['f'] = nut_bat_dau['chi_phi'] + nut_bat_dau['h']

        # Hàng đợi ưu tiên
        open_set = [(nut_bat_dau['f'], 0, nut_bat_dau)]
        counter = 1

        # Lưu chi phí tốt nhất cho mỗi trạng thái
        chi_phi_tot_nhat = {}

        so_nut_mo_rong = 0

        while open_set:
            f_val, _, current = heapq.heappop(open_set)
            so_nut_mo_rong += 1

            # Kiểm tra đã thăm hết các thành phố chưa
            if len(current['da_tham']) == self.n:
                # Quay về thành phố bắt đầu để hoàn thành chu trình
                tong_chi_phi = current['chi_phi'] + self.khoang_cach[current['hien_tai']][thanh_pho_bat_dau]
                if tong_chi_phi < float('inf'):
                    duong_di_hoan_chinh = current['duong_di'] + [thanh_pho_bat_dau]
                    thoi_gian_chay = time.time() - thoi_gian_bat_dau
                    return {
                        'duong_di': duong_di_hoan_chinh,
                        'chi_phi': tong_chi_phi,
                        'so_nut_mo_rong': so_nut_mo_rong,
                        'thoi_gian': thoi_gian_chay,
                        'thanh_cong': True
                    }
                continue

            # Sinh các trạng thái kề (các thành phố chưa thăm)
            for tp_tiep_theo in range(self.n):
                if tp_tiep_theo not in current['da_tham']:
                    tap_moi = frozenset(list(current['da_tham']) + [tp_tiep_theo])
                    chi_phi_moi = current['chi_phi'] + self.khoang_cach[current['hien_tai']][tp_tiep_theo]

                    # Bỏ qua nếu không có đường đi
                    if chi_phi_moi >= float('inf'):
                        continue

                    trang_thai_key = (tp_tiep_theo, tap_moi)

                    # Cắt tỉa nếu tìm thấy đường tốt hơn
                    if trang_thai_key in chi_phi_tot_nhat and chi_phi_tot_nhat[trang_thai_key] <= chi_phi_moi:
                        continue

                    chi_phi_tot_nhat[trang_thai_key] = chi_phi_moi
                    h_moi = ham_h(tap_moi, tp_tiep_theo)

                    nut_ke = {
                        'hien_tai': tp_tiep_theo,
                        'da_tham': tap_moi,
                        'duong_di': current['duong_di'] + [tp_tiep_theo],
                        'chi_phi': chi_phi_moi,
                        'h': h_moi
                    }
                    nut_ke['f'] = chi_phi_moi + h_moi

                    heapq.heappush(open_set, (nut_ke['f'], counter, nut_ke))
                    counter += 1

        return {
            'thanh_cong': False,
            'so_nut_mo_rong': so_nut_mo_rong,
            'thoi_gian': time.time() - thoi_gian_bat_dau
        }

    def in_loi_giai(self, ket_qua):
        """In kết quả đường đi TSP"""
        if not ket_qua['thanh_cong']:
            print(f"Không tìm thấy chu trình Hamilton!")
            print(f"Số nút đã mở rộng: {ket_qua['so_nut_mo_rong']}")
            print(f"Thời gian: {ket_qua['thoi_gian']:.2f} giây")
            return

        print(f"✓ Tìm thấy hành trình tối ưu!")
        print(f"Tổng khoảng cách: {ket_qua['chi_phi']:.2f}")
        print(f"Số nút đã mở rộng: {ket_qua['so_nut_mo_rong']}")
        print(f"Thời gian: {ket_qua['thoi_gian']:.2f} giây")
        print("\nHành trình:")

        # Chuyển đổi số thành tên thành phố
        hanh_trinh = " -> ".join(self.ten_thanh_pho[tp] for tp in ket_qua['duong_di'])
        print(hanh_trinh)
        print(f"\nĐộ dài hành trình: {len(ket_qua['duong_di'])} thành phố (kể cả quay về)")

# Chạy thử nghiệm
if __name__ == "__main__":
    import random

    print("GIẢI BÀI TOÁN NGƯỜI GIAO HÀNG (TSP) BẰNG A*\n")

    # Tạo ma trận khoảng cách cho 6 thành phố
    so_thanh_pho = 6
    khoang_cach = [[0] * so_thanh_pho for _ in range(so_thanh_pho)]

    # Tạo khoảng cách ngẫu nhiên (đối xứng)
    print("Ma trận khoảng cách giữa các thành phố:")
    for i in range(so_thanh_pho):
        for j in range(i + 1, so_thanh_pho):
            dist = random.randint(10, 100)
            khoang_cach[i][j] = dist
            khoang_cach[j][i] = dist

    # In ma trận
    print("    " + " ".join(f"TP{i:3d}" for i in range(so_thanh_pho)))
    for i in range(so_thanh_pho):
        print(f"TP{i}: " + " ".join(f"{khoang_cach[i][j]:3d}" for j in range(so_thanh_pho)))

    print("\n" + "="*60)

    # Giải TSP
    tsp = TSP_AStar(khoang_cach)
    ten_tp = [f"Thành phố {i}" for i in range(so_thanh_pho)]
    tsp.dat_ten_thanh_pho(ten_tp)

    print("Đang giải TSP bằng A*...")
    ket_qua = tsp.giai_tsp(thanh_pho_bat_dau=0, heuristic='mst')
    tsp.in_loi_giai(ket_qua)

    # So sánh với vét cạn (chỉ cho số thành phố nhỏ)
    if so_thanh_pho <= 8:
        print("\n" + "="*60)
        print("So sánh với phương pháp vét cạn (brute force)...")

        chi_phi_min = float('inf')
        duong_di_tot_nhat = None

        for hoan_vi in permutations(range(1, so_thanh_pho)):
            duong_di = [0] + list(hoan_vi) + [0]
            chi_phi = sum(khoang_cach[duong_di[i]][duong_di[i+1]] for i in range(len(duong_di)-1))
            if chi_phi < chi_phi_min:
                chi_phi_min = chi_phi
                duong_di_tot_nhat = duong_di

        print(f"Vét cạn tìm được chi phí tối ưu: {chi_phi_min}")
        print(f"A* tìm được chi phí: {ket_qua['chi_phi']}")
        print(f"Chênh lệch: {abs(ket_qua['chi_phi'] - chi_phi_min):.2f}")

        if ket_qua['chi_phi'] == chi_phi_min:
            print("✓ A* tìm được lời giải tối ưu!")
        else:
            print("✗ A* chưa tìm được lời giải tối ưu (có thể do heuristic không admissible)")

GIẢI BÀI TOÁN NGƯỜI GIAO HÀNG (TSP) BẰNG A*

Ma trận khoảng cách giữa các thành phố:
    TP  0 TP  1 TP  2 TP  3 TP  4 TP  5
TP0:   0  14  96  12  34  55
TP1:  14   0  51  32  50  62
TP2:  96  51   0  72  33 100
TP3:  12  32  72   0  22  86
TP4:  34  50  33  22   0  32
TP5:  55  62 100  86  32   0

Đang giải TSP bằng A*...
✓ Tìm thấy hành trình tối ưu!
Tổng khoảng cách: 215.00
Số nút đã mở rộng: 69
Thời gian: 0.00 giây

Hành trình:
Thành phố 0 -> Thành phố 3 -> Thành phố 1 -> Thành phố 2 -> Thành phố 4 -> Thành phố 5 -> Thành phố 0

Độ dài hành trình: 7 thành phố (kể cả quay về)

So sánh với phương pháp vét cạn (brute force)...
Vét cạn tìm được chi phí tối ưu: 215
A* tìm được chi phí: 215
Chênh lệch: 0.00
✓ A* tìm được lời giải tối ưu!
